# 💰 Wallet & 🏷️ Tagging

Two specialized sub-clients:
- **WalletClient** — idea and strength points (economic layer)
- **TaggingClient** — classification system with user + system tags

## Setup

> ⚠️ Wallet requires authentication. Tagging is fully client-side and works without auth.

In [ ]:
(async () => {
  const { InterfacerClient, Token } = await import('https://esm.sh/@dyne/interfacer-client');
  const { SANDBOX_CONFIG } = await import('/files/config.js');
  const client = new InterfacerClient(SANDBOX_CONFIG);
  console.log('💰 Wallet ready');
  console.log('🏷️ Tagging ready (client-side, no auth needed)');
  console.log('🔐 Authenticated:', client.isAuthenticated());
  console.log('🌐 Wallet URL:', client.config.walletUrl || '(not configured)');
})();


## 💰 Wallet: Idea & Strength Points

Two token types track community engagement:

| Token | Purpose |
|---|---|
| `Token.Idea` | Innovation / contribution credits |
| `Token.Strengths` | Community reputation / expertise |

The wallet is a **signed REST API** at `walletUrl`.

In [ ]:
(async () => {
  const agentId = client.store.getItem('authId');
  if (agentId) {
    try {
      const ideaBalance = await client.wallet.getBalance(agentId, Token.Idea);
      const strengthBalance = await client.wallet.getBalance(agentId, Token.Strengths);
      
      console.log('💰 Balances:');
      console.log('  💡 Idea:', ideaBalance);
      console.log('  💪 Strengths:', strengthBalance);
    } catch (err) {
      console.warn('⚠️ Balance fetch failed:', err.message);
    }
    
    // Balance at a specific point in time
    try {
      const lastWeek = Date.now() - 7 * 24 * 60 * 60 * 1000;
      const lastWeekBalance = await client.wallet.getBalanceAt(agentId, Token.Idea, lastWeek);
      console.log('\n📅 Idea balance (7 days ago):', lastWeekBalance);
    } catch (err) {
      console.warn('⚠️ Historical balance failed:', err.message);
    }
  } else {
    console.log('⚠️ Not authenticated — wallet requires auth.');
  }
})();


## 💰 Add Points & Trends

In [ ]:
(async () => {
  if (agentId) {
    // Add points (signed request)
    try {
      await client.wallet.addPoints(agentId, Token.Idea, 10);
      console.log('✅ Added 10 Idea points to', agentId.substring(0, 8) + '...');
    } catch (err) {
      console.warn('⚠️ addPoints failed:', err.message);
    }
    
    // Get trend (percentage change since period start)
    try {
      const lastWeek = Date.now() - 7 * 24 * 60 * 60 * 1000;
      const trend = await client.wallet.getTrend(agentId, Token.Idea, lastWeek);
      console.log('📈 Idea trend (7d):', trend.toFixed(1) + '%');
    } catch (err) {
      console.warn('⚠️ Trend fetch failed:', err.message);
    }
  }
})();


## 🏷️ Tagging: Classification System

The tagging system stores tags in the `classifiedAs` array on EconomicResources.
Tags use **prefixes** to distinguish user-entered values from system-derived metadata.

### Prefix Architecture

| Prefix | Type | Example |
|---|---|---|
| `tag-` | User tag | `tag-3d-printing` |
| `category-` | Product category | `category-electronics` |
| `power-compat-` | Power compatibility | `power-compat-usb-c` |
| `powerreq-` | Power requirement (range) | `powerreq-ge-100` |
| `recyclability-` | Recyclability (range) | `recyclability-ge-70` |
| `env-energy-` | Energy consumption (range) | `env-energy-le-50` |
| `env-co2-` | CO2 footprint (range) | `env-co2-ge-10` |
| `replicability-` | Replicability level | `replicability-easy` |
| `service-type-` | Service type | `service-type-assembly` |
| `availability-` | Availability | `availability-online` |

> 💡 Tagging is **fully client-side** — no API calls required!

## 1. User Tags — Normalize & Extract

In [ ]:
// Normalize raw user input to canonical tag-<slug> format
const normalized = client.tagging.normalizeUserTags([
  '3D Printing',
  'Open Source Hardware',
  'IoT',
  '  HANDMADE  ',  // whitespace + case
  'Électronique',   // unicode → ascii slug
]);
console.log('📋 Normalized user tags:');
normalized.forEach(t => console.log('  ', t));
// Create a single user tag
const tag = client.tagging.userTag('3D Printing');
console.log('\n🏷️ Single tag:', tag);
// Check if a tag is a user tag
console.log('  isUserTag?', client.tagging.isUserTag(tag));
console.log('  stripped:', client.tagging.stripUserTagPrefix(tag));


## 2. Extract & Filter Tags from ClassifiedAs

In [ ]:
// Simulate a classifiedAs array from a resource
const classifiedAs = [
  'tag-3d-printing',
  'tag-iot',
  'category-electronics',
  'powerreq-ge-100',
  'powerreq-le-150',
  'recyclability-ge-70',
  'recyclability-le-80',
  'manufacturable-true',
  'repairability-available',
  'legacy-plain-tag',  // Unprefixed legacy tag
];
// Extract user-visible tags (strip system prefixes)
const userTags = client.tagging.extractUserTagValues(classifiedAs);
console.log('👤 User-visible tags:', userTags);
// Check system tags
console.log('\n🔍 Tag classification:');
for (const t of classifiedAs) {
  console.log(`  ${t}: user=${client.tagging.isUserTag(t)}, system=${client.tagging.isSystemTag(t)}`);
}


## 3. Numeric Range Tags

For quantitative properties (power, recyclability, energy, CO2),
the system generates monotonic range tags (`ge-` = greater-or-equal, `le-` = less-or-equal).

In [ ]:
// Recyclability: 75% falls in the 70-80% bucket
const recyclabilityTags = client.tagging.monotonicRangeTags(
  'recyclability',
  75,
  [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
);
console.log('♻️ Recyclability 75% tags:', recyclabilityTags);
// Power requirement: 120W
const powerReqTags = client.tagging.monotonicRangeTags(
  'powerreq',
  120,
  [0, 10, 50, 100, 150, 200, 500, 1000]
);
console.log('⚡ Power 120W tags:', powerReqTags);
// Range filter from min/max (for search UI)
const filterTags = client.tagging.rangeFilterTags(
  'powerreq',
  50,   // min
  200,  // max
  [0, 10, 50, 100, 150, 200, 500, 1000]
);
console.log('🔍 Power filter [50-200]:', filterTags);


## 4. Derived Product & Service Filters

Auto-generate all system tags from structured filter metadata.

In [ ]:
// Product filter → all derived system tags
const productTags = client.tagging.derivedProductFilterTags({
  categories: ['Electronics', 'IoT'],
  powerCompatibility: ['USB-C', 'Battery'],
  replicability: ['Easy'],
  powerRequirementW: 120,
  recyclabilityPct: 75,
  energyKwh: 45,
  co2Kg: 12,
  repairability: true,
});
console.log('📦 Derived product tags (' + productTags.length + ' total):');
productTags.forEach(t => console.log('  ', t));
// Service filter
const serviceTags = client.tagging.derivedServiceFilterTags({
  serviceType: ['Assembly', 'Consulting'],
  availability: ['Online', 'In-Person'],
});
console.log('\n🛠️ Derived service tags:');
serviceTags.forEach(t => console.log('  ', t));


## 5. Tag Array Operations

In [ ]:
// Merge tag arrays (deduplicates)
const tags1 = ['tag-a', 'tag-b'];
const tags2 = ['tag-b', 'tag-c'];
const merged = client.tagging.mergeTags(tags1, tags2);
console.log('🔀 Merged:', merged);
// Remove tags with specific prefixes
const allTags = ['tag-hello', 'category-electronics', 'powerreq-ge-100', 'tag-world'];
const cleaned = client.tagging.removeTagsWithPrefixes(allTags, ['category', 'powerreq']);
console.log('🧹 Cleaned (no category/powerreq):', cleaned);
// Check if a tag has a known prefix
console.log('\n🔍 Prefix checks:');
console.log('  tag-hello isPrefixedTag(tag):', 
  client.tagging.removeTagsWithPrefixes(['tag-hello'], ['tag']).length === 0);
// Slugify a value
console.log('\n🔤 Slugify examples:');
console.log('  "Hello World!" →', client.tagging.slugifyTagValue('Hello World!'));
console.log('  "3D Printing" →', client.tagging.slugifyTagValue('3D Printing'));
console.log('  "Électronique" →', client.tagging.slugifyTagValue('Électronique'));


## 6. Tagging Constants

The client exposes all predefined options and thresholds.

In [ ]:
console.log('📋 Tagging constants:');
console.log('  TAG_PREFIX:', client.tagging.TAG_PREFIX);
console.log('  SYSTEM_TAG_PREFIXES:', client.tagging.SYSTEM_TAG_PREFIXES);
console.log('\n  PRODUCT_CATEGORY_OPTIONS:', client.tagging.PRODUCT_CATEGORY_OPTIONS.join(', '));
console.log('  POWER_COMPATIBILITY_OPTIONS:', client.tagging.POWER_COMPATIBILITY_OPTIONS.join(', '));
console.log('  REPLICABILITY_OPTIONS:', client.tagging.REPLICABILITY_OPTIONS.join(', '));
console.log('  SERVICE_TYPE_OPTIONS:', client.tagging.SERVICE_TYPE_OPTIONS.join(', '));
console.log('  AVAILABILITY_OPTIONS:', client.tagging.AVAILABILITY_OPTIONS.join(', '));
console.log('\n  RECYCLABILITY_THRESHOLDS_PCT:', client.tagging.RECYCLABILITY_THRESHOLDS_PCT.join(', '));
console.log('  POWER_REQUIREMENT_THRESHOLDS_W:', client.tagging.POWER_REQUIREMENT_THRESHOLDS_W.join(', '));
console.log('  ENERGY_THRESHOLDS_KWH:', client.tagging.ENERGY_THRESHOLDS_KWH.join(', '));
console.log('  CO2_THRESHOLDS_KG:', client.tagging.CO2_THRESHOLDS_KG.join(', '));
console.log('\n  MANUFACTURABLE_TRUE_TAG:', client.tagging.MANUFACTURABLE_TRUE_TAG);
console.log('  REPAIRABILITY_AVAILABLE_TAG:', client.tagging.REPAIRABILITY_AVAILABLE_TAG);


## 📋 API Reference

### WalletClient

| Method | Description |
|---|---|
| `getBalance(agentId, token)` | Current balance (Idea/Strengths) |
| `getBalanceAt(agentId, token, timestamp)` | Historical balance |
| `addPoints(agentId, token, amount)` | Add points (signed) |
| `getTrend(agentId, token, periodStartMs)` | % change since period start |

### TaggingClient (all client-side)

| Method | Description |
|---|---|
| `normalizeUserTags(tags)` | Raw input → canonical `tag-*` form |
| `extractUserTagValues(classifiedAs)` | Filter system tags, strip prefixes |
| `isUserTag(tag)` | Check if `tag-` prefixed |
| `isSystemTag(tag)` | Check if known system prefix |
| `userTag(raw)` | Create a single user tag |
| `slugifyTagValue(value)` | Slugify to lowercase hyphenated |
| `prefixedTag(prefix, value)` | Create `prefix-slug` tag |
| `monotonicRangeTags(prefix, value, thresholds)` | Range tags for a numeric value |
| `rangeFilterTags(prefix, min, max, thresholds)` | Range tags for a filter range |
| `derivedProductFilterTags(filters)` | All system tags from product metadata |
| `derivedServiceFilterTags(filters)` | All system tags from service metadata |
| `mergeTags(...lists)` | Merge + deduplicate tag arrays |
| `removeTagsWithPrefixes(tags, prefixes)` | Filter out prefix-matching tags |

## ✅ Summary
- ✅ Idea and Strength token balances
- ✅ Historical balance lookups and trends
- ✅ Adding points with signed requests
- ✅ User tag normalization and extraction
- ✅ System tag prefix architecture
- ✅ Numeric range tag generation (ge/le)
- ✅ Derived product and service filter tags
- ✅ Tag array merge and cleanup operations
- ✅ All predefined constants and threshold arrays

## 🎉 You've completed the Interfacer SDK documentation!

Check out `07_Import_and_Internals.ipynb` for GitHub/GitLab import and GraphQL internals.